# Feature engineering & Modeling
Last updated: _2022.06.08_

In [1]:
import pandas as pd

import matplotlib.pyplot as plt

In [2]:
df_data = pd.read_csv('output/dataset_churn_labeled_th8_20260609.csv')

In [3]:
print("There are {:,} rows and {:} columns in the raw data.".format(df_data.shape[0], df_data.shape[1]))
print("The dataset contains data from {:} to {:}.".format(df_data['summary_date'].min(), df_data['summary_date'].max()))

There are 545,488 rows and 54 columns in the raw data.
The dataset contains data from 2021-01-01 to 2022-12-31.


In [4]:
df_data['summary_date'] = pd.to_datetime(df_data['summary_date'])
df_data['registration_date'] = pd.to_datetime(df_data['registration_date'])
df_data['firstdeposit_createddate'] = pd.to_datetime(df_data['firstdeposit_createddate'])
df_data['is_churned'] = df_data['is_churned'].astype(int)

In [5]:
df_data.sort_values(['accountid', 'summary_date'], ascending=True, inplace=True)

## Feature engineering

In [6]:
# decided to use GBP as base
columns_to_keep = [
    'accountid',

    # date related
    'summary_date', 'registration_date', 'firstdeposit_createddate',
    
    # categorical
    'agp_brand', 'reporting_region', 'market', 'reg_prod', 'foundation_category', 'account_tier', 'category', 'activity_channel', 'secondary_product', 'bet_type', 'productgroup',
    
    # numerical
    'plc_turnover_gbp', 'plc_turnover_gbp_rcash', 'plc_turnover_gbp_bonus', 'plc_betcount',
    
    'stl_turnover_gbp', 'stl_turnover_gbp_rcash', 'stl_turnover_gbp_bonus',
    'stl_ggr_gbp', 'stl_ggr_gbp_rcash', 'stl_ggr_gbp_bonus', 
    
    #'stl_betting_duty', 'stl_licence_fee', 'stl_tax', 'stl_partnerfee',
    
    'stl_betcount',
    
    'deposit_gbp', 'withdrawal_gbp',
    
    'bonus_costs', 'realcash_costs', 'bonusfunds_costs', 'bonus_costs_accurrency', 'bonusfunds_costs_accurrency',

    'is_churned'
]

In [7]:
df_data = df_data[columns_to_keep].copy()

### Date

In [8]:
account_age = (
    df_data.groupby("accountid")
        .agg(
            registration_date=("registration_date", "first"),
            last_summary_date=("summary_date", "max")
        )
    )

account_age['account_age_in_days'] = (
    account_age["last_summary_date"] - account_age["registration_date"]
).dt.days

account_age = account_age.reset_index()
account_age.columns = ['accountid', 'registration_date', 'last_summary_date', 'account_age_in_days']
account_age = account_age[['accountid', 'registration_date', 'account_age_in_days']].copy() 

In [9]:
account_age[:5]

,accountid,registration_date,account_age_in_days
0,666989420,2020-10-03,684
1,666989923,2020-10-03,507
2,666990238,2020-10-03,471
3,666999616,2020-10-08,726
4,667005177,2020-10-10,562


In [10]:
time_to_first_deposit_age = (
    df_data.groupby("accountid")
        .agg(
            registration_date=("registration_date", "first"),
            first_deposit_date=("firstdeposit_createddate", "first")
        )
    )

time_to_first_deposit_age['time_to_first_deposit_age'] = (
    time_to_first_deposit_age["first_deposit_date"] - time_to_first_deposit_age["registration_date"]
).dt.days

time_to_first_deposit_age = time_to_first_deposit_age.reset_index()
time_to_first_deposit_age.columns = ['accountid', 'registration_registration', 'first_deposit_date', 'time_to_first_deposit_age']
time_to_first_deposit_age = time_to_first_deposit_age[['accountid', 'time_to_first_deposit_age']].copy() 

In [11]:
time_to_first_deposit_age[:5]

,accountid,time_to_first_deposit_age
0,666989420,0
1,666989923,0
2,666990238,0
3,666999616,0
4,667005177,374


### Numerical

In [12]:
numerical_columns = ['plc_turnover_gbp', 'plc_turnover_gbp_rcash', 'plc_turnover_gbp_bonus', 'plc_betcount',
    
    'stl_turnover_gbp', 'stl_turnover_gbp_rcash', 'stl_turnover_gbp_bonus', 'stl_ggr_gbp', 'stl_ggr_gbp_rcash', 'stl_ggr_gbp_bonus', 'stl_betcount',
    
    # not in description, columns most probably calculated from other columns
    #'stl_betting_duty', 'stl_licence_fee', 'stl_tax', 'stl_partnerfee',

    'deposit_gbp', 'withdrawal_gbp',
    
    'bonus_costs', 'realcash_costs', 'bonusfunds_costs', 'bonus_costs_accurrency', 'bonusfunds_costs_accurrency'
]

In [13]:
df_numerical = df_data[list(['accountid']) + numerical_columns].groupby('accountid', as_index=False).agg({
    'plc_turnover_gbp': ['min', 'mean', 'median', 'max', 'nunique'],
    'plc_turnover_gbp_rcash': ['min', 'mean', 'median', 'max', 'nunique'],
    'plc_turnover_gbp_bonus': ['min', 'mean', 'median', 'max', 'nunique'],
    'plc_betcount': ['min', 'mean', 'median', 'max', 'nunique'],
    
    'stl_turnover_gbp': ['min', 'mean', 'median', 'max', 'nunique'],
    'stl_turnover_gbp_rcash': ['min', 'mean', 'median', 'max', 'nunique'],
    'stl_turnover_gbp_bonus': ['min', 'mean', 'median', 'max', 'nunique'],
    'stl_ggr_gbp': ['min', 'mean', 'median', 'max', 'nunique'],
    'stl_ggr_gbp_rcash': ['min', 'mean', 'median', 'max', 'nunique'],
    'stl_ggr_gbp_bonus': ['min', 'mean', 'median', 'max', 'nunique'],
    'stl_betcount': ['min', 'mean', 'median', 'max', 'nunique'],

    'deposit_gbp': ['min', 'mean', 'median', 'max', 'nunique'],
    'withdrawal_gbp': ['min', 'mean', 'median', 'max', 'nunique'],
    
    'bonus_costs': ['min', 'mean', 'median', 'max', 'nunique'],
    'realcash_costs': ['min', 'mean', 'median', 'max', 'nunique'],
    'bonusfunds_costs': ['min', 'mean', 'median', 'max', 'nunique'],
    'bonus_costs_accurrency': ['min', 'mean', 'median', 'max', 'nunique'],
    'bonusfunds_costs_accurrency': ['min', 'mean', 'median', 'max', 'nunique']
})

df_numerical.columns = [
    col[0] if col[1] == '' else f'{col[0]}_{col[1]}'
    for col in df_numerical.columns
]

In [14]:
df_numerical[:5]

,accountid,plc_turnover_gbp_min,plc_turnover_gbp_mean,plc_turnover_gbp_median,plc_turnover_gbp_max,plc_turnover_gbp_nunique,plc_turnover_gbp_rcash_min,plc_turnover_gbp_rcash_mean,plc_turnover_gbp_rcash_median,plc_turnover_gbp_rcash_max,...,bonus_costs_accurrency_min,bonus_costs_accurrency_mean,bonus_costs_accurrency_median,bonus_costs_accurrency_max,bonus_costs_accurrency_nunique,bonusfunds_costs_accurrency_min,bonusfunds_costs_accurrency_mean,bonusfunds_costs_accurrency_median,bonusfunds_costs_accurrency_max,bonusfunds_costs_accurrency_nunique
0,666989420,0.0,0.756415,0.50,15.00,43,0.0,0.703585,0.5,6.00,...,0.0,0.052830,0.0,10.0,3,0.0,0.052830,0.0,10.0,3
1,666989923,0.0,23.413187,5.00,426.00,40,0.0,21.800733,0.0,426.00,...,0.0,1.611722,0.0,25.0,6,0.0,1.611722,0.0,25.0,6
2,666990238,0.0,9.396947,5.00,110.00,14,0.0,7.087786,0.0,100.00,...,0.0,2.309160,0.0,17.0,8,0.0,2.309160,0.0,17.0,8
3,666999616,0.0,21.793820,9.50,779.00,42,0.0,21.793820,9.5,779.00,...,0.0,0.000000,0.0,0.0,1,0.0,0.000000,0.0,0.0,1
4,667005177,0.0,42.962111,6.25,425.17,114,0.0,41.885986,5.0,425.17,...,0.0,1.076125,0.0,25.0,7,0.0,1.076125,0.0,25.0,7


### Ohe

In [15]:
columns_to_be_ohed = ['agp_brand', 'reporting_region', 'market', 'reg_prod', 'foundation_category', 'account_tier', 'category', 'activity_channel', 'secondary_product', 'bet_type', 'productgroup']

In [16]:
df_data_ohe = pd.get_dummies(df_data[list(['accountid']) + columns_to_be_ohed], columns=columns_to_be_ohed, sparse=False, drop_first=True)

In [17]:
df_data_ohe[:5]

,accountid,agp_brand_Parimatch UK,reporting_region_United Kingdom,market_Ireland,market_Parimatch UK,reg_prod_Sports,reg_prod_eGaming,foundation_category_Paint,foundation_category_Plaster,account_tier_AVERAGE,...,bet_type_Others,bet_type_RnG,bet_type_Single,productgroup_Casino,productgroup_Live Casino,productgroup_Lottery,productgroup_Overall,productgroup_Sportsbook,productgroup_Virtual Sports,productgroup_eSports
6397,666989420,True,True,False,True,True,False,True,False,False,...,False,False,False,False,False,False,False,True,False,False
53760,666989420,True,True,False,True,True,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
53761,666989420,True,True,False,True,True,False,True,False,False,...,False,False,True,False,False,False,False,True,False,False
137109,666989420,True,True,False,True,True,False,True,False,False,...,False,False,True,False,False,False,False,True,False,False
143199,666989420,True,True,False,True,True,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False


In [18]:
agg_dict = {col: ["min", "max"] for col in df_data_ohe.columns[1:]}

In [19]:
df_data_ohe_grouped = df_data_ohe.groupby('accountid', as_index=False).agg(agg_dict)

df_data_ohe_grouped.columns = [
    col[0] if col[1] == '' else f'{col[0]}_{col[1]}'
    for col in df_data_ohe_grouped.columns
]

### Merging

In [20]:
df_training_set = pd.merge(
    account_age,
    time_to_first_deposit_age,
    how='left',
    on='accountid'
)

In [21]:
df_training_set = pd.merge(
    df_training_set,
    df_data_ohe_grouped,
    how='left',
    on='accountid'
)

In [22]:
df_training_set = pd.merge(
    df_training_set,
    df_numerical,
    how='left',
    on='accountid'
)

In [23]:
df_training_set.sort_values('accountid', inplace=True)

In [24]:
y = df_data[['accountid', 'registration_date', 'is_churned']].groupby('accountid', as_index=False).agg({
    'registration_date': 'first',
    'is_churned': 'max'
}).sort_values('accountid', inplace=False)

In [25]:
df_training_set[:5]

,accountid,registration_date,account_age_in_days,time_to_first_deposit_age,agp_brand_Parimatch UK_min,agp_brand_Parimatch UK_max,reporting_region_United Kingdom_min,reporting_region_United Kingdom_max,market_Ireland_min,market_Ireland_max,...,bonus_costs_accurrency_min,bonus_costs_accurrency_mean,bonus_costs_accurrency_median,bonus_costs_accurrency_max,bonus_costs_accurrency_nunique,bonusfunds_costs_accurrency_min,bonusfunds_costs_accurrency_mean,bonusfunds_costs_accurrency_median,bonusfunds_costs_accurrency_max,bonusfunds_costs_accurrency_nunique
0,666989420,2020-10-03,684,0,True,True,True,True,False,False,...,0.0,0.052830,0.0,10.0,3,0.0,0.052830,0.0,10.0,3
1,666989923,2020-10-03,507,0,True,True,True,True,False,False,...,0.0,1.611722,0.0,25.0,6,0.0,1.611722,0.0,25.0,6
2,666990238,2020-10-03,471,0,True,True,True,True,False,False,...,0.0,2.309160,0.0,17.0,8,0.0,2.309160,0.0,17.0,8
3,666999616,2020-10-08,726,0,True,True,True,True,False,False,...,0.0,0.000000,0.0,0.0,1,0.0,0.000000,0.0,0.0,1
4,667005177,2020-10-10,562,374,True,True,True,True,False,False,...,0.0,1.076125,0.0,25.0,7,0.0,1.076125,0.0,25.0,7


In [26]:
y[:5]

,accountid,registration_date,is_churned
0,666989420,2020-10-03,0
1,666989923,2020-10-03,0
2,666990238,2020-10-03,0
3,666999616,2020-10-08,0
4,667005177,2020-10-10,0


In [27]:
df_training = df_training_set[df_training_set['registration_date'] < '2022-01-01'][[i for i in df_training_set.columns if i not in ['accountid', 'registration_date']]]
df_test = df_training_set['2022-01-01' <= df_training_set['registration_date']][[i for i in df_training_set.columns if i not in ['accountid', 'registration_date']]]

In [28]:
df_training.shape, df_test.shape

((1323, 276), (677, 276))

In [29]:
df_training_y = y[y['registration_date'] < '2022-01-01'][[i for i in y.columns if i not in ['accountid', 'registration_date']]]
df_test_y = y['2022-01-01' <= y['registration_date']][[i for i in y.columns if i not in ['accountid', 'registration_date']]]

In [30]:
df_training_y.shape, df_test_y.shape

((1323, 1), (677, 1))

In [31]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, roc_auc_score, precision_recall_curve, auc

In [32]:
def calculate_metrics(title, y_true, y_pred):
    precision, recall, thresholds = precision_recall_curve(y_true, y_pred)
    pr_auc_trapz = auc(recall, precision)

    print("{:}\n- Accuracy: {:>15.4f}\n- Precision: {:>14.4f}\n- Recall: {:>17.4f}\n- F1 score: {:>15.4f}\n- Cohen-kappa score: {:>6.4f}\n- ROC AuC: {:>16.4f}\n- PR AuC: {:>17.4f}".format(
        title,
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred),
        recall_score(y_true, y_pred),
        f1_score(y_true, y_pred),
        cohen_kappa_score(y_true, y_pred),
        roc_auc_score(y_true, y_pred),
        pr_auc_trapz
    ))

## Decision tree

In [33]:
from sklearn.tree import DecisionTreeClassifier

In [34]:
clf = DecisionTreeClassifier()

In [35]:
clf.fit(df_training, df_training_y['is_churned'])

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [36]:
df_training_y['prediction'] = clf.predict(df_training)
df_test_y['prediction'] = clf.predict(df_test)

In [37]:
calculate_metrics("Training set:", df_training_y['is_churned'], df_training_y['prediction'])

Training set:
- Accuracy:          1.0000
- Precision:         1.0000
- Recall:            1.0000
- F1 score:          1.0000
- Cohen-kappa score: 1.0000
- ROC AuC:           1.0000
- PR AuC:            1.0000


In [38]:
calculate_metrics("Test set:", df_test_y['is_churned'], df_test_y['prediction'])

Test set:
- Accuracy:          0.3072
- Precision:         1.0000
- Recall:            0.0042
- F1 score:          0.0085
- Cohen-kappa score: 0.0026
- ROC AuC:           0.5021
- PR AuC:            0.8485


## RandomForest

In [39]:
from sklearn.ensemble import RandomForestClassifier

In [40]:
clf = RandomForestClassifier()

In [41]:
clf.fit(df_training, df_training_y['is_churned'])

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [42]:
df_training_y['prediction'] = clf.predict(df_training)
df_test_y['prediction'] = clf.predict(df_test)

In [43]:
calculate_metrics("Training set:", df_training_y['is_churned'], df_training_y['prediction'])

Training set:
- Accuracy:          1.0000
- Precision:         1.0000
- Recall:            1.0000
- F1 score:          1.0000
- Cohen-kappa score: 1.0000
- ROC AuC:           1.0000
- PR AuC:            1.0000


In [44]:
calculate_metrics("Test set:", df_test_y['is_churned'], df_test_y['prediction'])

Test set:
- Accuracy:          0.3545
- Precision:         0.8864
- Recall:            0.0828
- F1 score:          0.1515
- Cohen-kappa score: 0.0370
- ROC AuC:           0.5293
- PR AuC:            0.8036


## GradientBoosting

In [45]:
from sklearn.ensemble import GradientBoostingClassifier

In [46]:
clf = GradientBoostingClassifier()

In [47]:
clf.fit(df_training, df_training_y['is_churned'])

,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"init init: estimator or 'zero', default=NoneAn estimator object that is used to co

In [48]:
df_training_y['prediction'] = clf.predict(df_training)
df_test_y['prediction'] = clf.predict(df_test)

In [49]:
calculate_metrics("Training set:", df_training_y['is_churned'], df_training_y['prediction'])

Training set:
- Accuracy:          0.9138
- Precision:         0.8729
- Recall:            0.9784
- F1 score:          0.9227
- Cohen-kappa score: 0.8261
- ROC AuC:           0.9104
- PR AuC:            0.9313


In [50]:
calculate_metrics("Test set:", df_test_y['is_churned'], df_test_y['prediction'])

Test set:
- Accuracy:          0.3072
- Precision:         1.0000
- Recall:            0.0042
- F1 score:          0.0085
- Cohen-kappa score: 0.0026
- ROC AuC:           0.5021
- PR AuC:            0.8485
